# Chain-of-Verification (CoVe)

Large Language Models are fluent — sometimes fluent enough to state a wrong fact with the same confidence as a correct one. Reflection and Reflexion (see the two companion notebooks in this folder) fight this by having the model critique its *own* answer in one more pass. But there's a well-known failure mode with that approach: if the model didn't know a fact was wrong the first time, simply re-reading its own answer rarely makes the error jump out. The model is prone to confirming its own bias.

**Chain-of-Verification (CoVe)** — introduced by Dhuliawala et al. (2023) and popularized as one of the canonical agentic architectures — fixes this by decomposing verification into small, independent, checkable questions, and by answering each of those questions **without the original (possibly wrong) answer in context**. Only after collecting these independently-sourced answers does the agent go back and revise the original response.

| Property | Value |
|---|---|
| Origin | Dhuliawala et al. (Meta), *Chain-of-Verification Reduces Hallucination in Large Language Models* (2023). [arXiv:2309.11495](https://arxiv.org/abs/2309.11495) |

### Definition
**Chain-of-Verification** is an agentic pattern in which a model's baseline answer is fact-checked by a set of atomic verification questions that are each answered *independently* (i.e., in a fresh LLM call that never sees the baseline answer), and the baseline answer is then revised using only the results of that independent verification. It trades one extra round of LLM calls for a large reduction in unverified hallucinations.

### High-level Workflow
1. **Baseline Response Generation** — the agent answers the user's query directly, in one pass. This is the same failure-prone single-shot generation any LLM would produce; it may contain confident-sounding but incorrect atomic facts.
2. **Plan Verification Questions** — the agent re-reads its own baseline answer and extracts a list of *independent, atomic* verification questions — one per checkable factual claim. Structured output (a Pydantic list of questions) keeps this machine-parsable.
3. **Execute Verifications Independently** — each verification question is sent to the LLM **on its own**, with no memory of the baseline answer or the other questions. This decoupling is the core trick of CoVe: it prevents the model from just rubber-stamping what it already said.
4. **Generate Final Verified Response** — the agent is shown the baseline answer *and* the full list of (question, independent answer) pairs, and asked to produce a corrected final answer — fixing or removing any claim that the independent verification contradicted.

### When to Use / Applications
* **Fact-heavy Q&A** — biography summaries, "list N facts about X" style queries, historical/geographical/scientific claims — anywhere a single paragraph packs in several independently-checkable atomic facts.
* **Long-form generation with citations or specifics** — dates, counts, names, statistics — where a plausible-sounding wrong number is worse than an admission of uncertainty.
* **Any RAG-adjacent generation step** where an extra grounding pass is cheap relative to the cost of a hallucination reaching the user.

### Strengths & Weaknesses
* **Strengths:**
  * **Breaks self-confirmation bias.** Because verification questions are answered without the baseline in context, the model can't just agree with itself — this is the key advantage over vanilla Reflection.
  * **Decomposition helps.** Atomic, independent questions are individually much easier for an LLM to get right than a dense paragraph of claims.
  * **Traceable.** Every correction in the final answer can be traced back to a specific verification question and its answer — useful for debugging and for building user trust.
* **Weaknesses:**
  * **Cost.** CoVe issues `1 (baseline) + 1 (planning) + N (verification questions) + 1 (final revision)` LLM calls — noticeably more expensive/slower than a single pass or even simple Reflection.
  * **Still bounded by the model's knowledge.** If the model is wrong about a fact *and* would independently answer the verification question the same wrong way, CoVe won't catch it — this pattern fixes inconsistency-style hallucination, not gaps in world knowledge.
  * **Question quality matters.** If the planning step produces vague or non-atomic questions, the independent-answer step degrades back toward simple self-critique.

## Phase 0: Setup

**What we are going to do:**
We initialize the LLM via this repo's shared `helpers` factory — never a direct provider client — so the notebook automatically uses Groq on Windows / Databricks on macOS, matching every other LangGraph-phase notebook in this repo.

In [ ]:
# ============ SETUP: IMPORTS AND LLM INITIALIZATION ============

from typing import List

from pydantic import BaseModel, Field

from helpers import get_llm

llm = get_llm()

print("LLM initialized via helpers.get_llm()")

## Phase 1: Baseline Response

**What we are going to do:**
We ask the agent a query that is deliberately prone to subtly wrong specific facts: *"List the key facts about the Eiffel Tower."* A dense, fact-listing query like this is exactly the shape of prompt where an LLM tends to blend correct general knowledge with a few confidently-wrong specifics (an incorrect height, a wrong architect detail, a slightly-off completion year, etc.). We deliberately do **not** correct anything at this stage — this is the raw, single-pass answer that CoVe exists to fact-check.

In [ ]:
# ============ STEP 1: GENERATE THE BASELINE RESPONSE ============

USER_QUERY = "List the key facts about the Eiffel Tower (height, architect, year completed, and location)."

baseline_prompt = (
    "Answer the user's question as a short list of specific, factual bullet points. "
    "Be precise and confident.\n\n"
    f"Question: {USER_QUERY}"
)

baseline_response = llm.invoke(baseline_prompt).content

print("=== BASELINE RESPONSE (unverified) ===")
print(baseline_response)

## Phase 2: Plan Verification Questions

**What we are going to do:**
We define a Pydantic schema (`VerificationPlan`) so the model returns a clean, machine-parsable list of atomic verification questions — one per checkable claim in the baseline response — instead of free-form prose. Each question should stand on its own (answerable without seeing the baseline text).

In [ ]:
# ============ STEP 2a: DEFINE STRUCTURED OUTPUT SCHEMAS ============

class VerificationPlan(BaseModel):
    """A set of independent, atomic fact-checking questions derived from a baseline answer."""

    questions: List[str] = Field(
        description=(
            "Independent, atomic verification questions — one per checkable factual claim "
            "in the baseline answer. Each question must be self-contained (answerable on its "
            "own, without needing the original answer as context) and must target exactly one fact."
        )
    )


class VerificationAnswer(BaseModel):
    """An independently-derived answer to a single verification question."""

    answer: str = Field(description="A concise, factual answer to the verification question.")


class VerifiedResponse(BaseModel):
    """The final, corrected answer produced after independent verification."""

    final_answer: str = Field(description="The revised answer, with any contradicted claims corrected.")
    corrections_made: List[str] = Field(
        description="A short list describing each correction made relative to the baseline answer. Empty if none."
    )


print("Pydantic schemas for VerificationPlan, VerificationAnswer, and VerifiedResponse defined.")

**Discussion of the Output:**
`VerificationPlan.questions` is the crux of the whole pattern: instead of one vague "is this correct?" prompt, we get N atomic, independently-answerable questions. Splitting the baseline answer's claims this finely is what makes the next step effective — a model that would gloss over a dense paragraph will often get a single, narrow question right.

In [ ]:
# ============ STEP 2b: PLAN THE VERIFICATION QUESTIONS ============

planning_llm = llm.with_structured_output(VerificationPlan)

planning_prompt = (
    "Below is a draft answer to a factual question. Break every distinct factual claim in it down "
    "into an independent, atomic verification question. Each question must be answerable entirely "
    "on its own, without seeing this draft answer.\n\n"
    f"Draft answer:\n{baseline_response}"
)

verification_plan = planning_llm.invoke(planning_prompt)

print("=== PLANNED VERIFICATION QUESTIONS ===")
for i, q in enumerate(verification_plan.questions, start=1):
    print(f"{i}. {q}")

## Phase 3: Answer Verification Questions Independently

**What we are going to do:**
This is the step that makes CoVe different from ordinary self-reflection. Each verification question gets its **own, fresh LLM call** — the baseline answer, the other questions, and any prior verification answers are never included in the prompt. This independence is deliberate: it stops the model from just re-confirming whatever it said the first time, and forces it to reason about each fact from scratch.

In [ ]:
# ============ STEP 3: ANSWER EACH QUESTION IN ISOLATION ============

verification_llm = llm.with_structured_output(VerificationAnswer)


def answer_independently(question: str) -> str:
    """Answer a single verification question with NO knowledge of the baseline answer."""
    isolated_prompt = (
        "Answer the following factual question concisely and accurately. "
        "Do not assume any prior context.\n\n"
        f"Question: {question}"
    )
    return verification_llm.invoke(isolated_prompt).answer


verification_results = [
    {"question": q, "independent_answer": answer_independently(q)}
    for q in verification_plan.questions
]

print("=== INDEPENDENT VERIFICATION RESULTS ===")
for i, result in enumerate(verification_results, start=1):
    print(f"Q{i}: {result['question']}")
    print(f"A{i}: {result['independent_answer']}\n")

**Discussion of the Output:**
Notice that each `answer_independently` call has no memory of the baseline response or of the other verification calls — every one of these `verification_llm.invoke(...)` calls is a brand-new context. If the baseline answer got a specific fact (say, the tower's height or completion year) subtly wrong, an independent, narrowly-scoped question about *just that fact* has a much better chance of surfacing the correct value than the model would have gotten by simply re-reading its own paragraph.

## Phase 4: Final Verified Response

**What we are going to do:**
Now — and only now — do we show the model the baseline answer *together with* every (question, independently-derived answer) pair. We ask it to produce a corrected final answer, fixing or removing any claim that the independent verification contradicted, and to explicitly list what it changed.

In [ ]:
# ============ STEP 4: REVISE THE BASELINE USING VERIFICATION RESULTS ============

revision_llm = llm.with_structured_output(VerifiedResponse)

verification_block = "\n".join(
    f"- Q: {r['question']}\n  Independently verified A: {r['independent_answer']}"
    for r in verification_results
)

revision_prompt = (
    "You previously gave the following baseline answer to a user's question. Some of its claims "
    "have since been checked independently — the results are listed below. Produce a corrected "
    "final answer that fixes or removes any claim contradicted by the independent verification, "
    "and keeps everything that was confirmed. List every correction you made.\n\n"
    f"User's original question: {USER_QUERY}\n\n"
    f"Baseline answer:\n{baseline_response}\n\n"
    f"Independent verification results:\n{verification_block}"
)

verified_response = revision_llm.invoke(revision_prompt)

print("=== FINAL VERIFIED RESPONSE ===")
print(verified_response.final_answer)
print("\n=== CORRECTIONS MADE ===")
if verified_response.corrections_made:
    for c in verified_response.corrections_made:
        print(f"- {c}")
else:
    print("(No corrections were necessary — every claim was confirmed by independent verification.)")

### Baseline vs. Verified — Side by Side

**What we are going to do:**
Print the baseline and the verified answers next to each other, along with the correction list, so the effect of the CoVe pass is concretely visible rather than just asserted.

In [ ]:
# ============ COMPARE BASELINE VS. VERIFIED OUTPUT ============

print("#" * 70)
print("# BEFORE (baseline, single-pass, unverified)")
print("#" * 70)
print(baseline_response)

print("\n" + "#" * 70)
print("# AFTER (Chain-of-Verification applied)")
print("#" * 70)
print(verified_response.final_answer)

print("\n" + "#" * 70)
print("# WHAT CHANGED")
print("#" * 70)
if verified_response.corrections_made:
    for c in verified_response.corrections_made:
        print(f"- {c}")
else:
    print("No corrections were necessary.")

**Discussion of the Output:**
Whatever the underlying model got wrong in the baseline pass — a mis-stated height, an incorrect year, a garbled attribution — should now show up explicitly in the `WHAT CHANGED` list, traceable back to one of the independent verification Q&A pairs from Phase 3. That traceability (this specific correction came from this specific independently-answered question) is the main practical advantage CoVe has over asking a model to "double check your answer" in a single extra turn.

## Optional: The Same Flow as a LangGraph

**What we are going to do:**
The four phases above are inherently a linear pipeline (`baseline -> plan -> verify (fan-out) -> revise`), so a plain sequential Python flow — as written above — is enough. For consistency with the other notebooks in this folder (which use `StateGraph`), here is the equivalent LangGraph wiring using a single state dict threaded through four nodes. It is not re-executed below; it documents how you would productionize this as a graph with retries/streaming/checkpointing.

In [ ]:
# ============ OPTIONAL: EQUIVALENT LANGGRAPH WIRING (for reference) ============

from typing import Optional, TypedDict

from langgraph.graph import END, StateGraph


class CoVeState(TypedDict):
    user_query: str
    baseline_response: Optional[str]
    verification_plan: Optional[List[str]]
    verification_results: Optional[list]
    final_answer: Optional[str]
    corrections_made: Optional[List[str]]


def baseline_node(state: CoVeState) -> dict:
    resp = llm.invoke(
        "Answer the user's question as a short list of specific, factual bullet points. "
        "Be precise and confident.\n\n"
        f"Question: {state['user_query']}"
    ).content
    return {"baseline_response": resp}


def plan_node(state: CoVeState) -> dict:
    plan = llm.with_structured_output(VerificationPlan).invoke(
        "Break every distinct factual claim in the draft answer below into an independent, "
        "atomic verification question, each answerable on its own.\n\n"
        f"Draft answer:\n{state['baseline_response']}"
    )
    return {"verification_plan": plan.questions}


def verify_node(state: CoVeState) -> dict:
    results = [
        {"question": q, "independent_answer": answer_independently(q)}
        for q in state["verification_plan"]
    ]
    return {"verification_results": results}


def revise_node(state: CoVeState) -> dict:
    block = "\n".join(
        f"- Q: {r['question']}\n  Independently verified A: {r['independent_answer']}"
        for r in state["verification_results"]
    )
    revised = llm.with_structured_output(VerifiedResponse).invoke(
        "Produce a corrected final answer that fixes or removes any claim contradicted by the "
        "independent verification below, and list every correction made.\n\n"
        f"User's original question: {state['user_query']}\n\n"
        f"Baseline answer:\n{state['baseline_response']}\n\n"
        f"Independent verification results:\n{block}"
    )
    return {"final_answer": revised.final_answer, "corrections_made": revised.corrections_made}


cove_graph_builder = StateGraph(CoVeState)
cove_graph_builder.add_node("generate_baseline", baseline_node)
cove_graph_builder.add_node("plan_verifications", plan_node)
cove_graph_builder.add_node("execute_verifications", verify_node)
cove_graph_builder.add_node("revise_response", revise_node)

cove_graph_builder.set_entry_point("generate_baseline")
cove_graph_builder.add_edge("generate_baseline", "plan_verifications")
cove_graph_builder.add_edge("plan_verifications", "execute_verifications")
cove_graph_builder.add_edge("execute_verifications", "revise_response")
cove_graph_builder.add_edge("revise_response", END)

cove_app = cove_graph_builder.compile()

print("CoVe LangGraph compiled: generate_baseline -> plan_verifications -> execute_verifications -> revise_response -> END")

# Example invocation (not run here to avoid duplicate LLM calls):
# result = cove_app.invoke({"user_query": USER_QUERY})

## Key Takeaways

* **Chain-of-Verification (CoVe)** turns fact-checking into a 4-step pipeline: **baseline answer -> plan atomic verification questions -> answer each question independently -> revise the baseline using only the verification results**.
* The defining trick is **independence**: verification questions are answered in fresh LLM calls that never see the original (possibly wrong) answer, so the model can't simply confirm its own prior claim. This is the key improvement over vanilla Reflection, where the critique step sees — and is anchored by — the original output.
* Decomposing a dense, fact-heavy answer into small, atomic questions makes each individual fact easier for the model to get right, even when the original paragraph blended a few wrong specifics in with mostly-correct ones.
* The cost is real: CoVe needs `1 + 1 + N + 1` LLM calls instead of 1, so it's best reserved for fact-heavy generation where a hallucinated specific (a wrong date, height, name, or statistic) is costly to let through.
* Like Reflection and Reflexion, CoVe is still bounded by the model's own knowledge — it corrects *inconsistency* between what the model states and what it independently believes, not gaps in what it knows at all.
* **Layer:** This pattern belongs in **Layer B3 — Reasoning and Reflection**, alongside `01_Reflection_Agents.ipynb` and `02_Reflexion_Agents.ipynb` in this same folder — all three are single/multi-agent self-improvement loops that trade extra LLM calls for higher answer quality.